# Week 2 Mini Project: House Price Prediction Model

**Topic:** Supervised Learning — Linear Regression  
**Objective:** Predict house prices, evaluate the model using the R² score, and plot predicted vs. actual values.

Dataset: `yasserh/housing-prices-dataset` from Kaggle via `kagglehub`.

> Run the cells from top to bottom. The first run may require internet access and Kaggle authentication, depending on your environment.


## 1. Install required libraries
If needed, run the following in a notebook cell. In Google Colab/Jupyter, remove the `#` at the start to install packages.


In [ ]:
# %pip install kagglehub[pandas-datasets] pandas scikit-learn matplotlib seaborn


## 2. Load the dataset
The adapter can infer the dataset file when `file_path` is omitted. If that fails, inspect the Kaggle dataset files and set `file_path` to the exact CSV filename.


In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from kagglehub import KaggleDatasetAdapter

file_path = "Housing.csv"  # If this filename is not found, replace it with the exact CSV filename in the dataset.

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yasserh/housing-prices-dataset",
    file_path,
)

print("First five records:")
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


## 3. Inspect and prepare the data
The target column is expected to be `price`. This cell checks that it exists, removes rows with missing target values, and prepares numeric and categorical features. Missing feature values are imputed and categorical columns are one-hot encoded inside a pipeline.


In [ ]:
target_column = "price"
if target_column not in df.columns:
    raise ValueError(
        f"Target column '{target_column}' was not found. Available columns: {df.columns.tolist()}. "
        "Check that you loaded the correct housing dataset and update target_column if necessary."
    )

df = df.dropna(subset=[target_column]).copy()
X = df.drop(columns=[target_column])
y = pd.to_numeric(df[target_column], errors="coerce")

valid_target = y.notna()
X = X.loc[valid_target]
y = y.loc[valid_target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of rows used:", len(X))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Missing values in target after cleaning:", y.isna().sum())


## 4. Split data and build the Linear Regression model
An 80:20 train-test split is used. The preprocessing steps are fitted only on the training data to reduce data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

transformers = []
if numeric_features:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    transformers.append(("numeric", numeric_pipeline, numeric_features))
if categorical_features:
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    transformers.append(("categorical", categorical_pipeline, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Model training completed.")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 5. Evaluate the model
R² is the main metric requested in the Week 2 mini-project. MAE and MSE are included as additional regression metrics. A higher R² is generally better; R² can be negative if the model performs worse than predicting the test-set mean.


In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:,.2f}")
print(f"Mean Squared Error (MSE): {mse:,.2f}")


## 6. Plot predicted vs. actual prices
Points closer to the diagonal reference line indicate predictions closer to actual prices.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
lower = min(float(y_test.min()), float(y_pred.min()))
upper = max(float(y_test.max()), float(y_pred.max()))
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.xlabel("Actual House Price")
plt.ylabel("Predicted House Price")
plt.title("Actual vs. Predicted House Prices — Linear Regression")
plt.tight_layout()
plt.show()


## 7. View a sample of predictions
This table compares actual and predicted prices for the first 10 test records.


In [ ]:
results = pd.DataFrame({
    "Actual Price": y_test.reset_index(drop=True),
    "Predicted Price": y_pred,
})
results["Prediction Error"] = results["Actual Price"] - results["Predicted Price"]
display(results.head(10).round(2))


## Conclusion
A Linear Regression model was trained to predict house prices. The test-set R² score indicates how much variation in the target prices is explained by the model. The scatter plot compares actual prices with predictions; points closer to the diagonal line indicate better predictions.

**Submission note:** Run every notebook cell and save the notebook with the output and graph visible before submitting. Confirm the dataset filename if loading fails.
